# DSA4213 Assignment 1 — Data preparation


Prepare **one shared dataset** for the n-gram, RNN, LSTM and transformer models. 

**How to run:** Upload to google colab. Change `PROJECT_ROOT` to a suitable folder. 

**Corpus:** WikiText-2


## 1. Corpus choice and experimental scope
**WikiText-2 (`wikitext-2-raw-v1`) is chosen because it is relatively small and manageable computationally for this project.** 

**Scope:** It contains Wikipedia articles on different topics and has existing train/validation/test splits. 

Sources: [dataset card](https://huggingface.co/datasets/Salesforce/wikitext), [original paper](https://arxiv.org/abs/1609.07843), [Datasets loading documentation](https://huggingface.co/docs/datasets/loading). 

## 2. Install dependencies

In [1]:
%pip -q install datasets huggingface_hub numpy pandas

## 3. Mount Drive and create paths


In [2]:
from google.colab import drive
from pathlib import Path
import json, hashlib, re, unicodedata, sys, platform, subprocess
from collections import Counter
from datetime import datetime, timezone
import importlib.metadata as metadata
import numpy as np
import pandas as pd
from IPython.display import display

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/DSA4213_Assignment1')
RAW_DIR = PROJECT_ROOT / 'data/raw'
PROCESSED_ROOT = PROJECT_ROOT / 'data/processed/lm'
CONFIG_DIR = PROJECT_ROOT / 'configs'
for folder in [RAW_DIR, PROCESSED_ROOT, CONFIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

def write_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

print('Project:', PROJECT_ROOT)

Mounted at /content/drive
Project: /content/drive/MyDrive/DSA4213_Assignment1


## 4. Fix shared preprocessing settings
Retain at most 9,996 ordinary tokens plus four special tokens:

| Token | ID | Purpose |
|---|---:|---|
| PAD | 0 | Fill unused positions; never score these targets |
| UNK | 1 | Represent excluded/unseen words; score normally |
| BOS | 2 | Start a row; input only |
| EOS | 3 | End a row; score normally |



In [3]:
SETTINGS = {
    'dataset_id': 'Salesforce/wikitext',
    'dataset_config': 'wikitext-2-raw-v1',
    'max_vocab_size': 10000,
    'min_frequency': 2,
    'block_length': 64,
    'seed': 42,
    'embedding_dimension': 100,
    'normalization': 'NFKC; curly apostrophes to straight; lowercase; collapse whitespace',
    'text_unit': 'each nonempty original row, including headings',
    'deduplication': 'normalized exact text; first within split; test > validation > train',
    'special_ids': {'PAD': 0, 'UNK': 1, 'BOS': 2, 'EOS': 3},
    'input_only_ids': [0, 2],
    'reset_state_each_block': True,
    'tokenizer_version': 'unicode_regex_v1',
}
PAD, UNK, BOS, EOS = 0, 1, 2, 3
SPLITS = ['train', 'validation', 'test']
assert SETTINGS['max_vocab_size'] > 4 and SETTINGS['block_length'] > 0
display(pd.DataFrame([(k, str(v)) for k, v in SETTINGS.items()], columns=['Setting', 'Value']))

,Setting,Value
0,dataset_id,Salesforce/wikitext
1,dataset_config,wikitext-2-raw-v1
2,max_vocab_size,10000
3,min_frequency,2
4,block_length,64
5,seed,42
6,embedding_dimension,100
7,normalization,NFKC; curly apostrophes to straight; lowercase...
8,text_unit,"each nonempty original row, including headings"
9,deduplication,normalized exact text; first within split; tes...


## 5. Load and preserve the original data

In [4]:
from datasets import load_dataset, load_from_disk
from huggingface_hub import HfApi

RAW_PATH = RAW_DIR / SETTINGS['dataset_config']
SOURCE_PATH = RAW_DIR / (SETTINGS['dataset_config'] + '_source.json')
if (RAW_PATH / 'dataset_dict.json').exists():
    raw_data = load_from_disk(str(RAW_PATH))
    source = json.loads(SOURCE_PATH.read_text()) if SOURCE_PATH.exists() else {
        'dataset_id': SETTINGS['dataset_id'], 'dataset_config': SETTINGS['dataset_config'],
        'revision': None, 'note': 'Existing snapshot; original revision and download date unknown.'
    }
    print('Loaded saved original dataset.')
elif RAW_PATH.exists():
    raise RuntimeError(f'Possibly incomplete snapshot at {RAW_PATH}. Inspect/rename that folder before retrying.')
else:
    revision = HfApi().dataset_info(SETTINGS['dataset_id']).sha
    raw_data = load_dataset(SETTINGS['dataset_id'], SETTINGS['dataset_config'], revision=revision)
    raw_data.save_to_disk(str(RAW_PATH))
    source = {
        'dataset_id': SETTINGS['dataset_id'], 'dataset_config': SETTINGS['dataset_config'],
        'revision': revision, 'downloaded_utc': datetime.now(timezone.utc).isoformat(),
        'source_url': 'https://huggingface.co/datasets/Salesforce/wikitext'
    }
assert set(raw_data) == set(SPLITS)
assert source['dataset_id'] == SETTINGS['dataset_id']
assert source['dataset_config'] == SETTINGS['dataset_config']
raw_hashes = {}
for split in SPLITS:
    assert 'text' in raw_data[split].column_names
    digest = hashlib.sha256()
    for row in raw_data[split]:
        assert isinstance(row['text'], str)
        digest.update((json.dumps(row['text'], ensure_ascii=False) + '\n').encode('utf-8'))
    raw_hashes[split] = digest.hexdigest()
if 'raw_text_sha256' in source:
    assert source['raw_text_sha256'] == raw_hashes, 'Raw snapshot changed; investigate before proceeding.'
source['raw_text_sha256'] = raw_hashes
source['raw_rows'] = {s: len(raw_data[s]) for s in SPLITS}
write_json(SOURCE_PATH, source)
print('Recorded revision:', source.get('revision'))

Loaded saved original dataset.
Recorded revision: None


## 6. Inspect the original text
Counts are **rows**.

In [5]:
display(pd.DataFrame([
    {'split': s, 'raw_rows': len(raw_data[s]),
     'blank_rows': sum(not r['text'].strip() for r in raw_data[s])}
    for s in SPLITS
]))
shown = 0
for row_id, row in enumerate(raw_data['train']):
    if row['text'].strip():
        print(f"Row {row_id}: {row['text'][:500]!r}\n")
        shown += 1
        if shown == 3:
            break

,split,raw_rows,blank_rows
0,train,36718,12951
1,validation,3760,1299
2,test,4358,1467


Row 1: ' = Valkyria Chronicles III = \n'

Row 3: ' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and f'

Row 4: " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series newcomers . Character designer Raita Honjou and composer Hitoshi Sakimoto both returned from previous entries , along with Valkyria Chronicles II director Takeshi Ozawa . A large 

## 7. Define normalization and tokenization
Preprocessing: We lowercase and normalize whitespace, retain punctuation/numbers, and do **not** remove stopwords, stem or lemmatize.

Contractions such as `don't` stay together; hyphens are separate tokens. Source spellings like `<UNK>` are tokenized as ordinary text.

Standardise the text while preserving the original lexicon. 

In [6]:
PREPROCESS_SOURCE = r'''import re
import unicodedata

TOKEN_PATTERN = r"[^\W\d_]+(?:'[^\W\d_]+)*|\d+(?:[.,]\d+)*|[^\w\s]|_"
TOKEN_RE = re.compile(TOKEN_PATTERN, flags=re.UNICODE)

def normalize_text(text):
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('’', "'").replace('‘', "'")
    return ' '.join(text.lower().split())

def tokenize(text):
    return TOKEN_RE.findall(normalize_text(text))
'''
exec(PREPROCESS_SOURCE)
SETTINGS['token_pattern'] = TOKEN_PATTERN
SETTINGS['preprocessing_source_sha256'] = hashlib.sha256(PREPROCESS_SOURCE.encode()).hexdigest()
examples = ["The cat sat.", "Don't stop!", "Café costs 3.50.", "state-of-the-art", "<UNK>"]
for text in examples:
    print(repr(text), '->', tokenize(text))
assert tokenize("Café costs 3.50.") == ['café', 'costs', '3.50', '.']
assert tokenize("Don’t") == ["don't"]
assert tokenize('<UNK>') == ['<', 'unk', '>']

'The cat sat.' -> ['the', 'cat', 'sat', '.']
"Don't stop!" -> ["don't", 'stop', '!']
'Café costs 3.50.' -> ['café', 'costs', '3.50', '.']
'state-of-the-art' -> ['state', '-', 'of', '-', 'the', '-', 'art']
'<UNK>' -> ['<', 'unk', '>']


## 8. Clean rows and audit duplicate text
Keep each nonempty row as one independent unit, including headings.

Keep the first exact normalized occurrence within a split. 

For overlap across splits, retain test first, then validation, then train, removing duplicates from lower-priority splits. This predetermined text-only policy preserves held-out material. This only concerns exact duplication. Near-duplicates and semantic overlaps are not removed. Every removal is logged by original row ID.

Deduplication is done to prevent data leakage between splits. 

In [7]:
clean = {}; removals = []; global_seen = set()
for split in ['test', 'validation', 'train']:
    local_seen = set(); kept = []
    for row_id, row in enumerate(raw_data[split]):
        text = normalize_text(row['text'])
        tokens = tokenize(text)
        reason = None
        if not tokens:
            reason = 'empty_or_token_empty'
        elif text in local_seen:
            reason = 'duplicate_within_split'
        elif text in global_seen:
            reason = 'duplicate_in_higher_priority_split'
        local_seen.add(text)
        if reason:
            removals.append({'split': split, 'row_id': row_id, 'reason': reason})
        else:
            kept.append({'row_id': row_id, 'text': text, 'tokens': tokens})
    clean[split] = kept
    global_seen.update(r['text'] for r in kept)
    assert kept, f'No retained rows in {split}'
for i, split in enumerate(SPLITS):
    texts = {r['text'] for r in clean[split]}
    assert len(texts) == len(clean[split])
    for other in SPLITS[i + 1:]:
        assert texts.isdisjoint(r['text'] for r in clean[other])
removal_table = pd.DataFrame(removals, columns=['split', 'row_id', 'reason'])
display(removal_table.groupby(['split', 'reason']).size().rename('removed_rows').reset_index())

,split,reason,removed_rows
0,test,duplicate_within_split,99
1,test,empty_or_token_empty,1467
2,train,duplicate_in_higher_priority_split,266
3,train,duplicate_within_split,2060
4,train,empty_or_token_empty,12951
5,validation,duplicate_in_higher_priority_split,64
6,validation,duplicate_within_split,100
7,validation,empty_or_token_empty,1299


## 9. Build the vocabulary from training text only
Count tokens only in retained training rows. Sort by decreasing frequency, with alphabetical tie-breaking. Validation/test words do not influence which tokens enter the vocabulary. Rare and unseen words map to UNK. Later models must use this exact mapping.

In [8]:
counts = Counter(t for r in clean['train'] for t in r['tokens'])
ranked = sorted(counts, key=lambda token: (-counts[token], token))
lexical = [t for t in ranked if counts[t] >= SETTINGS['min_frequency']][:SETTINGS['max_vocab_size'] - 4]
id_to_token = ['<PAD>', '<UNK>', '<BOS>', '<EOS>'] + lexical
token_to_id = {t: i for i, t in enumerate(id_to_token)}
assert len(token_to_id) == len(id_to_token)
assert not any(t in counts for t in id_to_token[:4])
print('Training token types before filtering:', len(counts))
print('Saved vocabulary size, including special tokens:', len(id_to_token))
display(pd.DataFrame([{'token': t, 'train_frequency': counts[t], 'id': token_to_id[t]} for t in lexical[:15]]))
encoded = {
    s: [{'row_id': r['row_id'], 'ids': [token_to_id.get(t, UNK) for t in r['tokens']]} for r in clean[s]]
    for s in SPLITS
}

Training token types before filtering: 65131
Saved vocabulary size, including special tokens: 10000


,token,train_frequency,id
0,the,130758,4
1,",",102590,5
2,.,84313,6
3,of,57022,7
4,and,50559,8
5,@,45592,9
6,in,45005,10
7,to,39520,11
8,a,36666,12
9,"""",28303,13


## 10. Summarize retained data and unknown words
UNK rate is the percentage of ordinary token occurrences mapped to UNK, before adding BOS/EOS/PAD. A high rate means perplexity is less informative because all the unknowns are collapsed into one bucket. 

In [9]:
summary_rows = []
for split in SPLITS:
    lengths = np.array([len(r['ids']) for r in encoded[split]])
    total = int(lengths.sum())
    unknown = sum(r['ids'].count(UNK) for r in encoded[split])
    summary_rows.append({
        'split': split, 'raw_rows': len(raw_data[split]), 'retained_rows': len(lengths),
        'removed_rows': len(raw_data[split]) - len(lengths), 'lexical_tokens': total,
        'unknown_tokens': unknown, 'unknown_pct': 100 * unknown / total,
        'median_row_tokens': float(np.median(lengths)), 'p95_row_tokens': float(np.percentile(lengths, 95)),
        'max_row_tokens': int(lengths.max()), 'vocab_size': len(id_to_token)
    })
summary = pd.DataFrame(summary_rows)
display(summary.round(2))

,split,raw_rows,retained_rows,removed_rows,lexical_tokens,unknown_tokens,unknown_pct,median_row_tokens,p95_row_tokens,max_row_tokens,vocab_size
0,train,36718,21441,15277,2115237,160711,7.60,90.0,252.0,746,10000
1,validation,3760,2297,1463,221408,20450,9.24,86.0,245.0,437,10000
2,test,4358,2792,1566,249905,25189,10.08,76.0,245.0,497,10000


## 11. Create next-token inputs and targets
For `the cat`, use `x = [BOS, the, cat]` and `y = [the, cat, EOS]`. The target is shifted one position ahead. Each target must be predicted using **only preceding tokens**.

Long rows are divided into blocks of at most 64 predictions, with right padding on the final block. Every lexical token and EOS is a target exactly once. 

In [10]:
def make_blocks(rows, block_length):
    count = sum((len(r['ids']) + 1 + block_length - 1) // block_length for r in rows)
    x = np.full((count, block_length), PAD, dtype=np.int32)
    y = np.full_like(x, PAD)
    manifest = []
    index = 0
    for row in rows:
        z = [BOS] + row['ids'] + [EOS]
        for offset in range(0, len(z) - 1, block_length):
            length = min(block_length, len(z) - 1 - offset)
            x[index, :length] = z[offset:offset + length]
            y[index, :length] = z[offset + 1:offset + 1 + length]
            manifest.append({'block_id': index, 'row_id': row['row_id'],
                             'offset': offset, 'valid_length': length})
            index += 1
    return x, y, pd.DataFrame(manifest)

blocks = {s: make_blocks(encoded[s], SETTINGS['block_length']) for s in SPLITS}
x, y, manifest = blocks['train']
length = int(manifest.iloc[0]['valid_length'])
display(pd.DataFrame({
    'input': [id_to_token[i] for i in x[0, :min(length, 12)]],
    'target': [id_to_token[i] for i in y[0, :min(length, 12)]]
}))

,input,target
0,<BOS>,=
1,=,valkyria
2,valkyria,chronicles
3,chronicles,iii
4,iii,=
5,=,<EOS>


## 12. Check alignment, boundaries and target counts


In [11]:
toy_x, toy_y, toy_m = make_blocks([{'row_id': 7, 'ids': [4, 5, 6, 7, 8]}], 3)
assert toy_x.tolist() == [[BOS, 4, 5], [6, 7, 8]]
assert toy_y.tolist() == [[4, 5, 6], [7, 8, EOS]]
for split in SPLITS:
    x, y, manifest = blocks[split]
    expected = sum(len(r['ids']) + 1 for r in encoded[split])
    assert int((y != PAD).sum()) == expected == int(manifest.valid_length.sum())
    assert not np.any(y == BOS)
    assert x.min() >= 0 and y.min() >= 0
    assert x.max() < len(id_to_token) and y.max() < len(id_to_token)
    rows_by_id = {r['row_id']: r['ids'] for r in encoded[split]}
    for block in manifest.itertuples(index=False):
        z = [BOS] + rows_by_id[block.row_id] + [EOS]
        i, a, length = block.block_id, block.offset, block.valid_length
        assert x[i, :length].tolist() == z[a:a + length]
        assert y[i, :length].tolist() == z[a + 1:a + length + 1]
        assert np.all(x[i, length:] == PAD) and np.all(y[i, length:] == PAD)
    print(f'{split}: {len(x):,} blocks; {expected:,} scored targets; checks passed.')

train: 46,109 blocks; 2,136,678 scored targets; checks passed.
validation: 4,845 blocks; 223,705 scored targets; checks passed.
test: 5,652 blocks; 252,697 scored targets; checks passed.


## 13. Save reusable files and provenance

In [12]:
signature = {'settings': SETTINGS, 'raw_text_sha256': raw_hashes}
version = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:16]
OUTPUT_DIR = PROCESSED_ROOT / version
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMPLETE = OUTPUT_DIR / 'COMPLETE.json'
common = {**SETTINGS, 'vocab_size': len(id_to_token), 'data_version': version,
          'processed_dir_relative': str(OUTPUT_DIR.relative_to(PROJECT_ROOT)),
          'predicted_ids': list(range(1, len(id_to_token))) }
common['predicted_ids'].remove(BOS)

def write_jsonl(path, rows):
    with Path(path).open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + '\n')

if COMPLETE.exists():
    saved = json.loads(COMPLETE.read_text())
    for filename, expected_hash in saved['sha256'].items():
        assert file_hash(OUTPUT_DIR / filename) == expected_hash, f'Changed file: {filename}'
    assert json.loads((OUTPUT_DIR / 'config.json').read_text()) == common
    print('Verified and reused completed data version:', version)
else:
    write_json(OUTPUT_DIR / 'config.json', common)
    write_json(OUTPUT_DIR / 'source.json', source)
    write_json(OUTPUT_DIR / 'vocab.json', {'token_to_id': token_to_id, 'id_to_token': id_to_token})
    (OUTPUT_DIR / 'preprocessing.py').write_text(PREPROCESS_SOURCE, encoding='utf-8')
    for split in SPLITS:
        write_jsonl(OUTPUT_DIR / f'{split}_tokens.jsonl', clean[split])
        write_jsonl(OUTPUT_DIR / f'{split}_encoded.jsonl', encoded[split])
        x, y, manifest = blocks[split]
        np.savez_compressed(OUTPUT_DIR / f'{split}_blocks.npz', x=x, y=y)
        manifest.to_csv(OUTPUT_DIR / f'{split}_manifest.csv', index=False)
    summary.to_csv(OUTPUT_DIR / 'data_summary.csv', index=False)
    removal_table.to_csv(OUTPUT_DIR / 'removed_rows.csv', index=False)
    prompts = ['the history of', 'the city is', 'during the first', 'the team won']
    write_json(OUTPUT_DIR / 'generation_prompts.json', [
        {'text': p, 'tokens': tokenize(p), 'ids': [token_to_id.get(t, UNK) for t in tokenize(p)]}
        for p in prompts])
    environment = {'python': sys.version, 'platform': platform.platform(),
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'packages': {p: metadata.version(p) for p in ['datasets', 'huggingface_hub', 'numpy', 'pandas']}}
    write_json(OUTPUT_DIR / 'environment.json', environment)
    (OUTPUT_DIR / 'requirements-data-lock.txt').write_text(
        ''.join(f'{p}=={v}\n' for p, v in environment['packages'].items()))
    freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
    (OUTPUT_DIR / 'environment-full.txt').write_text(freeze)
    hashes = {p.name: file_hash(p) for p in sorted(OUTPUT_DIR.iterdir()) if p.is_file() and p != COMPLETE}
    write_json(COMPLETE, {'data_version': version, 'sha256': hashes})
write_json(CONFIG_DIR / 'common.json', common)
print('Saved data folder:', OUTPUT_DIR)

Saved data folder: /content/drive/MyDrive/DSA4213_Assignment1/data/processed/lm/beacb3de63dc4261


## 14. Reload from disk and confirm completion


In [13]:
saved_vocab = json.loads((OUTPUT_DIR / 'vocab.json').read_text())
assert saved_vocab['id_to_token'] == id_to_token
completion = json.loads(COMPLETE.read_text())
for filename, expected_hash in completion['sha256'].items():
    assert file_hash(OUTPUT_DIR / filename) == expected_hash
for split in SPLITS:
    with np.load(OUTPUT_DIR / f'{split}_blocks.npz', allow_pickle=False) as saved:
        assert np.array_equal(saved['x'], blocks[split][0])
        assert np.array_equal(saved['y'], blocks[split][1])
display(pd.read_csv(OUTPUT_DIR / 'data_summary.csv').round(2))
print('DATA PREPARATION COMPLETE')
print('Version:', version)
print('Next notebook: load configs/common.json, vocab.json and the saved blocks.')
print('All models must use this same data version.')

,split,raw_rows,retained_rows,removed_rows,lexical_tokens,unknown_tokens,unknown_pct,median_row_tokens,p95_row_tokens,max_row_tokens,vocab_size
0,train,36718,21441,15277,2115237,160711,7.60,90.0,252.0,746,10000
1,validation,3760,2297,1463,221408,20450,9.24,86.0,245.0,437,10000
2,test,4358,2792,1566,249905,25189,10.08,76.0,245.0,497,10000


DATA PREPARATION COMPLETE
Version: beacb3de63dc4261
Next notebook: load configs/common.json, vocab.json and the saved blocks.
All models must use this same data version.
